In [1]:
import pandas as pd,numpy as np,json
from sklearn.metrics.pairwise import cosine_distances
import getpass,os
from langchain.chat_models import init_chat_model
from ai_patterns_mining import parse_json_safe,Config
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from time import time
from langchain.tools import tool
from pydantic import BaseModel, Field
import seaborn as sns
import matplotlib.pyplot as plt
from langchain.agents.structured_output import ToolStrategy
from tqdm import tqdm

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pattern_descriptions= json.load(open('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/outputs/prompt & rag/20251028_085918 - Run /extracted_patterns/pattern_summaries_dec_16.json'))

In [3]:
pattern_descriptions

[{'cluster_id': 'Advanced LLM Prompting',
  'pattern_summary': '**Pattern Description:** This pattern describes a comprehensive approach to designing, optimizing, and managing prompts for Large Language Models (LLMs) to achieve high-quality, controlled, and contextually relevant outputs across diverse and complex tasks. It addresses challenges such as output control, reasoning enhancement, bias mitigation, and multilingual performance. The solution involves strategically structuring prompts with templates, few-shot examples, or specific roles/styles, and leveraging techniques for automated prompt generation and optimization. Furthermore, it integrates external knowledge, pre-processing steps like translation, and mechanisms for task decomposition into sequential prompt chains. The pattern also emphasizes incorporating metacognitive cues for improved reasoning and integrating human-in-the-loop or automated feedback for iterative refinement and ambiguity resolution, ultimately leading to

In [4]:
# Build LLM helper to map None rows to one of the proposed new patterns
from langchain_core.prompts import ChatPromptTemplate
from typing import Optional
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")
    
llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)

In [5]:
class PatternMatchOutput(BaseModel):
    """Result from AI pattern matching (0–100 confidence scores)."""

    advanced_llm_prompting: int = Field(
        ...,
        description="0-100 score for pattern: Advanced LLM Prompting"
    )

    cross_lingual_llm_prompting: int = Field(
        ...,
        description="0-100 score for pattern: Cross-lingual LLM Prompting"
    )

    llm_based_multimodal_generative_prompting: int = Field(
        ...,
        description="0-100 score for pattern: LLM based Multimodal Generative Prompting"
    )

    integrating_external_knowledge_with_llm: int = Field(
        ...,
        description="0-100 score for pattern: Integrating External Knlowladge with LLM"
    )

    llm_context_management: int = Field(
        ...,
        description="0-100 score for pattern: LLM Context Management"
    )

    retrieval_augmented_generation_optimization_for_llms: int = Field(
        ...,
        description="0-100 score for pattern: Retrieval Augmented Generation(RAG) Optimization for LLMs"
    )

    llm_based_planning_iterative_optimizations_and_reasoning: int = Field(
        ...,
        description=(
            "0-100 score for pattern: "
            "LLM based Planning, Iterative Optimizations and ReAct, or Reasoning, Think Step by step, XoT"
        )
    )

    tool_use_for_llms: int = Field(
        ...,
        description="0-100 score for pattern: Tool Use for LLMs"
    )

    enhanced_user_intent_comprehension_with_llms: int = Field(
        ...,
        description="0-100 score for pattern: Enhanced User Intent Comprehension with LLMs"
    )

    modular_llm_agent_architectures: int = Field(
        ...,
        description="0-100 score for pattern: Modular LLM Agent Architectures"
    )

    explainable_ai_xai_techniques: int = Field(
        ...,
        description="0-100 score for pattern: Explainable AI (XAI) Techniques"
    )

    llm_kv_cache_optimization: int = Field(
        ...,
        description="0-100 score for pattern: LLM KV Cache Optimization"
    )

    structured_output_and_formatting_for_llms: int = Field(
        ...,
        description="0-100 score for pattern: Structured Output & Formatting for LLMs"
    )

    llms_for_recommender_systems: int = Field(
        ...,
        description="0-100 score for pattern: LLMs for Recommender Systems"
    )

    llm_agent_training_and_alignment: int = Field(
        ...,
        description="0-100 score for pattern: LLM Agent Training & Alignment"
    )

    llm_results_evaluation: int = Field(
        ...,
        description="0-100 score for pattern: LLM Results Evaluation"
    )

    reliable_transparent_and_augmented_llms: int = Field(
        ...,
        description="0-100 score for pattern: Reliable, Transparent, & Augmented LLMs"
    )

    llm_code_execution_for_precision: int = Field(
        ...,
        description="0-100 score for pattern: LLM Code Execution for Precision"
    )

    model_abstraction_pattern: int = Field(
        ...,
        description="0-100 score for pattern: Model Abstraction Pattern"
    )

    classical_models: int = Field(
        ...,
        description="0-100 score for pattern: Classical Models"
    )

    preprocessing_text_and_numerical_data: int = Field(
        ...,
        description="0-100 score for pattern: Preprocessing Text and Numerical Data"
    )

    none: int = Field(
        ...,
        description="0-100 score for pattern: None"
    )

    predicted_pattern: Optional[str] = Field(
        None,
        description=(
            "The pattern with the highest score. if there are multiple patterns with the same highest score, select the one that represents the most suitable pattern for the code. "
            "MUST be exactly one of the cluster_id values from the pattern descriptions."
        )
    )

In [6]:
cluster_id_map = {
    "advanced_llm_prompting": "Advanced LLM Prompting",

    "cross_lingual_llm_prompting": "Cross-lingual LLM Prompting",

    "llm_based_multimodal_generative_prompting":
        "LLM based Multimodal Generative Prompting",

    "integrating_external_knowledge_with_llm":
        "Integrating External Knlowladge with LLM",

    "llm_context_management": "LLM Context Management",

    "retrieval_augmented_generation_optimization_for_llms":
        "Retrieval Augmented Generation(RAG) Optimization for LLMs",

    "llm_based_planning_iterative_optimizations_and_reasoning":
        "LLM based Planning, Iterative Optimizations and ReAct, or Reasoning, Think Step by step, XoT",

    "tool_use_for_llms": "Tool Use for LLMs",

    "enhanced_user_intent_comprehension_with_llms":
        "Enhanced User Intent Comprehension with LLMs",

    "modular_llm_agent_architectures":
        "Modular LLM Agent Architectures",

    "explainable_ai_xai_techniques":
        "Explainable AI (XAI) Techniques",

    "llm_kv_cache_optimization":
        "LLM KV Cache Optimization",

    "structured_output_and_formatting_for_llms":
        "Structured Output & Formatting for LLMs",

    "llms_for_recommender_systems":
        "LLMs for Recommender Systems",

    "llm_agent_training_and_alignment":
        "LLM Agent Training & Alignment",

    "llm_results_evaluation":
        "LLM Results Evaluation",

    "reliable_transparent_and_augmented_llms":
        "Reliable, Transparent, & Augmented LLMs",

    "llm_code_execution_for_precision":
        "LLM Code Execution for Precision",

    "model_abstraction_pattern":
        "Model Abstraction Pattern",

    "classical_models":
        "Classical Models",

    "preprocessing_text_and_numerical_data":
        "Preprocessing Text and Numerical Data",

    "none": "None",
}


In [7]:
@tool
def code_scoring(code: str):
    """
    Analyze source code and judge.
    """
    return f"""
You are an expert AI systems architect and static code analyst.

Your task is to analyze the given source code and assign a confidence score (0–100)
for EACH AI design pattern listed below, based strictly on evidence found in the code.

====================
SCORING RULES (STRICT)
====================

1. Score MUST be an integer between 0 and 100.
2. Use 0 if the code shows NO meaningful evidence of the pattern.
3. Use low scores (1–30) only for weak or indirect hints.
4. Use medium scores (31–70) only when multiple clear elements of the pattern exist.
5. Use high scores (71–100) ONLY when the pattern is explicitly and strongly implemented.
6. Do NOT infer intent. Score ONLY what is observable in the code.
7. Do NOT reward naming alone (e.g., class names without behavior).
8. Multiple patterns may have non-zero scores.
9. Avoid score inflation. Most patterns should be 0 unless clearly present.

====================
NONE CLASS RULE
====================

- Assign a HIGH score to the "none" pattern ONLY IF:
  - The code does NOT meaningfully match any other listed AI pattern, AND
  - The code mainly consists of:
    - utility functions
    - interfaces
    - configuration
    - infrastructure
    - non-AI logic
- If any AI pattern is clearly present, the "none" score should be LOW.

====================
PATTERN DEFINITIONS
====================

Below is the authoritative JSON description of each pattern.
Use these definitions EXACTLY as reference w

hen scoring.

{json.dumps(pattern_descriptions, indent=4)}

====================
SOURCE CODE
====================

{code}

====================
OUTPUT FORMAT (MANDATORY)
====================

- Return ONLY a valid JSON object.
- Do NOT include explanations, comments, or markdown.
- JSON keys MUST exactly match the expected output schema.
- Every pattern MUST have a score.

Example format (structure only):
"""


In [8]:

agent = create_agent(
    llm,
    tools=[code_scoring],
    response_format=ToolStrategy(PatternMatchOutput),
    system_prompt="You are an expert AI patterns analyst. Given a description of a code and a pattern description, determine if the pattern matches the code. Give score from 0-100"
)
def pattern_matches_code(code_summary: str) -> str:
    msg = f"""Score the following code:
{code_summary}"""
    inputs = {"messages": [{"role": "user", "content": msg}]}
    resp = agent.invoke(inputs)
    if resp.get('structured_response') is None:
        print("Retrying due to None response...")
        return pattern_matches_code(code_summary)
    return resp['structured_response']

In [9]:
communities = pd.read_json("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/pattern_classification_verification_results_NN_v2_list.json")
random_seed = 42
random_sample = communities.sample(n=300, random_state=random_seed).reset_index(drop=True)

In [10]:
random_sample

,code file,predicted labels,llm predicted high level pattern,llm predicted l2 pattern,confidence score,gemini verification explaination,code summary
0,https://github.com/HasinthakaPiyumal/AI-Patter...,"LLM based Planning, Iterative Optimizations an...","LLM based Planning, Iterative Optimizations an...",Planning,9.0,The code defines a '_create_behavior' method t...,The code primarily implements a **Behavior Tre...
1,https://github.com/HasinthakaPiyumal/AI-Patter...,"Tool Use for LLMs: 0.6519\nLLM based Planning,...","LLM based Planning, Iterative Optimizations an...",Planning with Extrospective Reasoning,9.0,The code implements an IDM-based planner that ...,The code implements the Intelligent Driver Mod...
2,https://github.com/HasinthakaPiyumal/AI-Patter...,Tool Use for LLMs: 0.8384,Tool Use for LLMs,Tool Augmentation,9.0,The code defines a comprehensive set of financ...,This code implements a robust set of **feature...
3,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM Agent Training & Alignment: 0.3957\nRetrie...,None,None,1.0,The code implements a general deep learning tr...,This code implements a comprehensive deep lear...
4,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM based Multimodal Generative Prompting: 0.4578,None,None,1.0,The code snippet is a Python unit test for a d...,The code reveals a pattern of multi-modal sens...
...,...,...,...,...,...,...,...
295,https://github.com/HasinthakaPiyumal/AI-Patter...,Tool Use for LLMs: 0.5176,Tool Use for LLMs,Tool Augmentation,9.0,The code implements a sophisticated object tra...,This code implements a multi-object tracking s...
296,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM Results Evaluation: 0.3783\nRetrieval Augm...,LLM Results Evaluation,Round-Trip Consistency Filtering (for Syntheti...,9.0,The `ConfigOptimizer`'s `_evaluate_temperature...,This code implements an adaptive classificatio...
297,https://github.com/HasinthakaPiyumal/AI-Patter...,"Tool Use for LLMs: 0.5058\nReliable, Transpare...",Tool Use for LLMs,Tool Understanding (via Prompting),9.0,The code explicitly uses a detailed prompt to ...,This code demonstrates a pattern of leveraging...
298,https://github.com/HasinthakaPiyumal/AI-Patter...,Tool Use for LLMs: 0.5083\nLLM based Multimoda...,None,None,1.0,The code defines a class for an entity in a si...,This code defines an `Entity` class that imple...


In [11]:
pattern_tree_path = "results/community_classification_score_v1.csv"
if os.path.exists(pattern_tree_path):
    pattern_tree = pd.read_csv(pattern_tree_path)
else:
    pattern_tree= pd.DataFrame(columns=['code_file']+list(PatternMatchOutput.model_fields.keys())+['max score','sum score','code summary'])
for _, row in tqdm(random_sample.iterrows()):
    _file = row['code file'].replace('https://github.com/HasinthakaPiyumal/AI-Pattern-Mining-Project/blob/main/notebooks/result/repo_callgraph_clusters/','/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/repo_callgraph_clusters/')    
    if row['code file'] in pattern_tree['code_file'].values:
        continue
    with open(_file,'r') as cf:
        code_summary = cf.read()
    pattern_result = pattern_matches_code(code_summary).__dict__
    scores = np.array(list(pattern_result.values()))[:-1]
    scores = scores.astype(int)
    pattern_result['max score'] = int(np.max(scores))
    pattern_result['sum score'] = int(np.sum(scores))
    predicted_pattern_index = int(np.argmax(scores))
    pattern_result['code_file'] = row['code file']
    pattern_result['code summary'] = row['code summary']
    pattern_tree.loc[len(pattern_tree)] = pattern_result
    pattern_tree.to_csv(pattern_tree_path,index=False)

0it [00:00, ?it/s]

300it [27:25,  5.49s/it]


In [43]:
pattern_tree

,code_file,advanced_llm_prompting,cross_lingual_llm_prompting,multimodal_generative_prompting,external_knowledge_integration,llm_context_management,rag_optimization,planning_and_reasoning_patterns,tool_use_for_llms,user_intent_comprehension,...,reliable_transparent_augmented_llms,code_execution_for_precision,model_abstraction_pattern,classical_models,preprocessing_text_numerical_data,none,max score,sum score,predicted pattern,code summary
0,https://github.com/HasinthakaPiyumal/AI-Patter...,0,0,0,15,0,0,95,0,0,...,0,0,85,0,0,0,95,195,Unknown,The code primarily implements a **Behavior Tre...


In [ ]:
outputs/prompt & rag/20251028_085918 - Run/generated_code-v2/LLMs for Recommender Systems